# Module 1 Final Assessment - SOLUTION
### Instructor Reference | Data Analytics in Python

100 marks total. Part A: 5 mini-projects (70 marks). Part B: 10 standalone questions (30 marks).
Each solution favours idiomatic, efficient Python (`.get()`, comprehensions, `max(key=)`, set operators)
over brute-force loops, matching the grading intent of the paper.

## PART A - Mini-Projects (70 marks)

### Project 1 - Clinic Patient Triage Log `[14 marks]`

In [1]:
patients = [
    {"id":"P01","name":"Ahmad Raza","age":34,"temp":38.9,"dept":"General"},
    {"id":"P02","name":"Sara Khan","age":28,"temp":36.7,"dept":"General"},
    {"id":"P03","name":"Bilal Ahmed","age":51,"temp":39.4,"dept":"ICU"},
    {"id":"P04","name":"Zara Malik","age":45,"temp":38.2,"dept":"General"},
    {"id":"P05","name":"Hassan Ali","age":67,"temp":40.1,"dept":"ICU"},
    {"id":"P06","name":"Nida Butt","age":39,"temp":36.9,"dept":"General"},
    {"id":"P07","name":"Usman Sheikh","age":29,"temp":37.5,"dept":"General"},
    {"id":"P08","name":"Ayesha Nawaz","age":31,"temp":38.6,"dept":"ICU"},
]

#### Q1 - average temperature (manual sum, no libraries)

In [ ]:
def avg_temp(patients):
    total = sum(p["temp"] for p in patients)
    return round(total / len(patients), 2)

print(f"Average temp: {avg_temp(patients)}°C")

Average temp: 38.29°C


#### Q2 - fever patients (list comprehension)

In [ ]:
def fever_patients(patients, threshold=37.5):
    return [p["name"] for p in patients if p["temp"] > threshold]

print(fever_patients(patients))

['Ahmad Raza', 'Bilal Ahmed', 'Zara Malik', 'Hassan Ali', 'Ayesha Nawaz']


#### Q3 - group by department (plain for loop)

In [ ]:
def by_department(patients):
    result = {}
    for p in patients:
        dept = p["dept"]
        result.setdefault(dept, []).append(p["name"])
    return result

print(by_department(patients))

{'General': ['Ahmad Raza', 'Sara Khan', 'Zara Malik', 'Nida Butt', 'Usman Sheikh'], 'ICU': ['Bilal Ahmed', 'Hassan Ali', 'Ayesha Nawaz']}


#### Q4 - oldest patient via max(key=)

In [ ]:
def oldest_patient(patients):
    return max(patients, key=lambda p: p["age"])

op = oldest_patient(patients)
print(f"Oldest: {op['name']} ({op['age']})")

Oldest: Hassan Ali (67)


#### Q5 - `.get()` vs `direct lookup`

In [ ]:
vitals_extra = {"P01": "stable", "P05": "critical"}

for p in patients:
    # .get(pid, 'unknown') never raises KeyError, unlike patients_extra[pid]
    # which would crash for any patient not present in vitals_extra.
    status = vitals_extra.get(p["id"], "unknown")
    print(f"{p['name']:<15} status={status}")

Ahmad Raza      status=stable
Sara Khan       status=unknown
Bilal Ahmed     status=unknown
Zara Malik      status=unknown
Hassan Ali      status=critical
Nida Butt       status=unknown
Usman Sheikh    status=unknown
Ayesha Nawaz    status=unknown


#### Q6 - triage order by temperature descending

In [ ]:
def triage_order(patients):
    return [p["id"] for p in sorted(patients, key=lambda p: p["temp"], reverse=True)]

print(triage_order(patients))

['P05', 'P03', 'P01', 'P08', 'P04', 'P07', 'P06', 'P02']


#### Q7 - department summary: count + avg_temp per dept

In [ ]:
def department_summary(patients):
    depts = by_department(patients)   # reuse Q3
    summary = {}
    for dept, names in depts.items():
        temps = [p["temp"] for p in patients if p["dept"] == dept]
        summary[dept] = {"count": len(names), "avg_temp": round(sum(temps)/len(temps), 2)}
    return summary

print(department_summary(patients))

{'General': {'count': 5, 'avg_temp': 37.64}, 'ICU': {'count': 3, 'avg_temp': 39.37}}


### Project 2 - Mini E-Commerce Order Book `[14 marks]`

In [8]:
orders = [
    {"order_id":"O1001","customer":"Ahmad","items":[("Mouse",2,1200),("Keyboard",1,2500)]},
    {"order_id":"O1002","customer":"Sara","items":[("Monitor",1,32000)]},
    {"order_id":"O1003","customer":"Bilal","items":[("USB Hub",3,1800),("Mousepad",2,450)]},
    {"order_id":"O1004","customer":"Ahmad","items":[("Webcam",1,4500),("Headset",1,6800)]},
    {"order_id":"O1005","customer":"Zara","items":[("Laptop",1,85000)]},
    {"order_id":"O1006","customer":"Sara","items":[("Keyboard",2,2500),("Mouse",1,1200)]},
]
# each item tuple = (product_name, quantity, unit_price)

#### Q1 - total value of one order

In [ ]:
def order_total(order):
    return sum(qty * price for _, qty, price in order["items"])

print(order_total(orders[0]))   # 2*1200 + 1*2500 = 4900

4900


#### Q2 - totals for all orders (dict comprehension)

In [ ]:
def all_order_totals(orders):
    return {o["order_id"]: order_total(o) for o in orders}

print(all_order_totals(orders))

{'O1001': 4900, 'O1002': 32000, 'O1003': 6300, 'O1004': 11300, 'O1005': 85000, 'O1006': 6200}


#### Q3 - total spend per customer (accumulate across multiple orders)

In [ ]:
def customer_spend(orders):
    spend = {}
    for o in orders:
        spend[o["customer"]] = spend.get(o["customer"], 0) + order_total(o)
    return spend

print(customer_spend(orders))

{'Ahmad': 16200, 'Sara': 38200, 'Bilal': 6300, 'Zara': 85000}


#### Q4 - order with highest total value (max + key=, reusing order_total)

In [ ]:
biggest = max(orders, key=order_total)
print(f"Biggest order: {biggest['order_id']} = {order_total(biggest)}")

Biggest order: O1005 = 85000


#### Q5 - most purchased product by total quantity

In [ ]:
def most_purchased_product(orders):
    running_totals = {}
    for o in orders:
        for name, qty, _ in o["items"]:
            running_totals[name] = running_totals.get(name, 0) + qty
    return max(running_totals, key=running_totals.get)

print(most_purchased_product(orders))

Mouse


#### Q6 - unique products via set (no manual 'seen' check)

In [ ]:
all_names = [name for o in orders for name, _, _ in o["items"]]
unique_products = set(all_names)
print(unique_products)

{'Headset', 'Mousepad', 'Mouse', 'Keyboard', 'Webcam', 'Laptop', 'Monitor', 'USB Hub'}


#### Q7 - formatted receipt for one order

In [ ]:
def generate_receipt(order):
    lines = [f"Receipt - {order['order_id']} ({order['customer']})", "-"*40]
    for name, qty, price in order["items"]:
        line_total = qty * price
        lines.append(f"{name:<15}{qty:>3} x {price:>7,}  = {line_total:>8,}")
    lines.append("-"*40)
    lines.append(f"{'TOTAL':<15}{'':>3}   {'':>7}  = {order_total(order):>8,}")
    return "\n".join(lines)

print(generate_receipt(orders[0]))

Receipt - O1001 (Ahmad)
----------------------------------------
Mouse            2 x   1,200  =    2,400
Keyboard         1 x   2,500  =    2,500
----------------------------------------
TOTAL                         =    4,900


### Project 3 - Semester Exam Result Sheet `[14 marks]`

In [16]:
results = {
    "2023-CS-01": {"Math": 78, "Physics": 65, "Programming": 91},
    "2023-CS-02": {"Math": 88, "Physics": 82, "Programming": 95},
    "2023-CS-03": {"Math": 45, "Physics": 58, "Programming": 62},
    "2023-CS-04": {"Math": 92, "Physics": 89, "Programming": 84},
    "2023-CS-05": {"Math": 55, "Physics": 48, "Programming": 60},
    "2023-CS-06": {"Math": 70, "Physics": 73, "Programming": 68},
}

#### Q1 - average of one student's scores using .values()

In [ ]:
def student_average(scores):
    return sum(scores.values()) / len(scores)

print(round(student_average(results["2023-CS-01"]), 2))

78.0


#### Q2 - class averages (dict comprehension)

In [ ]:
def class_averages(results):
    return {sid: round(student_average(sc), 2) for sid, sc in results.items()}

print(class_averages(results))

{'2023-CS-01': 78.0, '2023-CS-02': 88.33, '2023-CS-03': 55.0, '2023-CS-04': 88.33, '2023-CS-05': 54.33, '2023-CS-06': 70.33}


#### Q3 - letter grade + grade report

In [ ]:
def assign_grade(avg):
    if avg >= 85: return "A"
    elif avg >= 70: return "B"
    elif avg >= 50: return "C"
    else: return "F"

averages = class_averages(results)
grade_report = {sid: assign_grade(avg) for sid, avg in averages.items()}
print(grade_report)

{'2023-CS-01': 'B', '2023-CS-02': 'A', '2023-CS-03': 'C', '2023-CS-04': 'A', '2023-CS-05': 'C', '2023-CS-06': 'B'}


#### Q4 - subject average with defensive `.get()`

In [ ]:
def subject_average(results, subject):
    # .get(subject, 0) prevents a KeyError if a student's record is missing
    # this subject; it silently treats a missing subject as 0, which is a
    # deliberate simplification - in real data you'd likely exclude that
    # student instead of scoring them 0.
    scores = [sc.get(subject, 0) for sc in results.values()]
    return round(sum(scores) / len(scores), 2)

print(subject_average(results, "Math"))

71.33


#### Q5 - top scorer in Programming via max(key=)

In [ ]:
top = max(results, key=lambda sid: results[sid]["Programming"])
print(f"{top}: {results[top]['Programming']}")

2023-CS-02: 95


#### Q6 - at-risk students (list comprehension)

In [ ]:
def at_risk_students(results, threshold=50):
    return [sid for sid, avg in class_averages(results).items() if avg < threshold]

print(at_risk_students(results))

[]


#### Q7 - grade distribution

In [ ]:
unique_grades = set(grade_report.values())
grade_distribution = {g: list(grade_report.values()).count(g) for g in unique_grades}
print(grade_distribution)

{'C': 2, 'B': 2, 'A': 2}


### Project 4 - DNA Sequence Quality Batch `[14 marks]`

In [25]:
import re

sequences = [
    "ATCGATCGATCG",
    "atcgGGCCatcg",
    "ATCGXATCG",       # invalid character X
    "GGGGCCCCATAT",
    "",                # empty
    "ATCG-ATCG",       # contains a hyphen
    "TTTTAAAACCCC",
    "ATCGATCGATCGATCGATCG",
]

#### Q1 - validity check via regex fullmatch

In [ ]:
def is_valid_dna(seq):
    return bool(seq) and bool(re.fullmatch(r"[ATCG]+", seq.upper()))

for s in sequences:
    print(f"{s!r:<25} valid={is_valid_dna(s)}")

'ATCGATCGATCG'            valid=True
'atcgGGCCatcg'            valid=True
'ATCGXATCG'               valid=False
'GGGGCCCCATAT'            valid=True
''                        valid=False
'ATCG-ATCG'               valid=False
'TTTTAAAACCCC'            valid=True
'ATCGATCGATCGATCGATCG'    valid=True


#### Q2 - GC content, safe for invalid/empty input

In [ ]:
def gc_content(seq):
    if not is_valid_dna(seq):
        return 0.0
    s = seq.upper()
    gc = s.count("G") + s.count("C")
    return round(gc / len(s) * 100, 1)

for s in sequences:
    print(f"{s!r:<25} GC={gc_content(s)}%")

'ATCGATCGATCG'            GC=50.0%
'atcgGGCCatcg'            GC=66.7%
'ATCGXATCG'               GC=0.0%
'GGGGCCCCATAT'            GC=66.7%
''                        GC=0.0%
'ATCG-ATCG'               GC=0.0%
'TTTTAAAACCCC'            GC=33.3%
'ATCGATCGATCGATCGATCG'    GC=50.0%


#### Q3 - filter valid sequences (list comprehension)

In [ ]:
valid_sequences = [s for s in sequences if is_valid_dna(s)]
print(valid_sequences)

['ATCGATCGATCG', 'atcgGGCCatcg', 'GGGGCCCCATAT', 'TTTTAAAACCCC', 'ATCGATCGATCGATCGATCG']


#### Q4 - GC report dict comprehension

In [ ]:
gc_report = {s: gc_content(s) for s in valid_sequences}
print(gc_report)

{'ATCGATCGATCG': 50.0, 'atcgGGCCatcg': 66.7, 'GGGGCCCCATAT': 66.7, 'TTTTAAAACCCC': 33.3, 'ATCGATCGATCGATCGATCG': 50.0}


#### Q5 - longest sequence via max(key=len)

In [ ]:
longest = max(valid_sequences, key=len)
print(f"Longest: {longest} (len={len(longest)})")

Longest: ATCGATCGATCGATCGATCG (len=20)


#### Q6 - count non-overlapping motif occurrences

In [ ]:
def count_motif(seq, motif="ATCG"):
    return len(re.findall(motif, seq.upper()))

print(count_motif("ATCGATCGATCGATCGATCG", "ATCG"))

5


#### Q7 - batch summary report

In [ ]:
def batch_report(sequences):
    valid = [s for s in sequences if is_valid_dna(s)]
    invalid = len(sequences) - len(valid)
    avg_gc = round(sum(gc_content(s) for s in valid) / len(valid), 1) if valid else 0.0
    return {"total": len(sequences), "valid_count": len(valid),
            "invalid_count": invalid, "avg_gc": avg_gc}

print(batch_report(sequences))

{'total': 8, 'valid_count': 5, 'invalid_count': 3, 'avg_gc': 53.3}


### Project 5 - Departmental Library Catalog `[14 marks]`

In [33]:
catalog = {
    "ISBN-001": {"title":"Intro to Algorithms","copies":3,"tags":{"CS","Math","Core"}},
    "ISBN-002": {"title":"Python for Data Analysis","copies":5,"tags":{"CS","Data","Elective"}},
    "ISBN-003": {"title":"Genomics Primer","copies":2,"tags":{"Bio","Data","Core"}},
    "ISBN-004": {"title":"Linear Algebra Basics","copies":0,"tags":{"Math","Core"}},
    "ISBN-005": {"title":"Statistics for Scientists","copies":4,"tags":{"Math","Data","Elective"}},
    "ISBN-006": {"title":"Machine Learning Intro","copies":1,"tags":{"CS","Data","Bio"}},
}

#### Q1 - out of stock titles

In [ ]:
def out_of_stock(catalog):
    return [b["title"] for b in catalog.values() if b["copies"] == 0]

print(out_of_stock(catalog))

['Linear Algebra Basics']


#### Q2 - books with a given tag (set membership)

In [ ]:
def books_with_tag(catalog, tag):
    return [b["title"] for b in catalog.values() if tag in b["tags"]]

print(books_with_tag(catalog, "Data"))

['Python for Data Analysis', 'Genomics Primer', 'Statistics for Scientists', 'Machine Learning Intro']


#### Q3 - union of all tags without a manual add-one-at-a-time loop

In [ ]:
all_tags = set().union(*(book["tags"] for book in catalog.values()))
print(all_tags)

{'Data', 'Math', 'Core', 'Bio', 'CS', 'Elective'}


#### Q4 - books tagged BOTH CS and Data (set intersection)

In [ ]:
def cs_and_data_books(catalog):
    return [b["title"] for b in catalog.values() if {"CS","Data"} <= b["tags"]]

print(cs_and_data_books(catalog))

['Python for Data Analysis', 'Machine Learning Intro']


#### Q5 - total copies via sum() + generator expression

In [ ]:
total_copies = sum(b["copies"] for b in catalog.values())
print(total_copies)

15


#### Q6 - restock needed (dict comprehension)

In [ ]:
def restock_needed(catalog, threshold=2):
    return {b["title"]: b["copies"] for b in catalog.values() if b["copies"] <= threshold}

print(restock_needed(catalog))

{'Genomics Primer': 2, 'Linear Algebra Basics': 0, 'Machine Learning Intro': 1}


#### Q7 - tag frequency across the catalog

In [ ]:
def tag_frequency(catalog):
    return {tag: sum(1 for b in catalog.values() if tag in b["tags"]) for tag in all_tags}

print(tag_frequency(catalog))

{'Data': 4, 'Math': 3, 'Core': 3, 'Bio': 2, 'CS': 3, 'Elective': 2}


## PART B - Standalone Questions (30 marks)

**Q36 - Efficiency check `[written]`**

`d.get(pid, 'Unknown')` is preferred over the `if/else` membership check.
It is a single expression rather than a 3-line branch, it can never raise a
`KeyError` even if written slightly wrong (e.g. forgetting the `in` check),
and it is the idiomatic, PEP-8-favoured way to provide a default for a
possibly-missing dict key. The `if pid in d: ... else: ...` form does the
same lookup twice internally (once for `in`, once for `d[pid]`), so
`.get()` is also marginally more efficient.

#### Q37 - chained string cleaning, no intermediate variables, no loop

In [ ]:
raw = "  Ahmad   RAZA  "
print(" ".join(raw.split()).title())

Ahmad Raza


#### Q38 - evens doubled, one comprehension

In [ ]:
nums = [4, 15, 8, 23, 16, 42, 7]
result = [n*2 for n in nums if n % 2 == 0]
print(result)

[8, 16, 32, 84]


#### Q39 - tuple unpacking

In [ ]:
record = ("P004", "Zara Malik", 45, 38.2)
pid, name, age, temp = record
print(f"{pid}: {name}, age {age}, temp {temp}°C")

P004: Zara Malik, age 45, temp 38.2°C


**Q40 - Set membership efficiency `[written]`**

Checking `x in some_set` is average-case **O(1)** because a set is backed by
a hash table - Python computes a hash of `x` and jumps almost directly to
its bucket. Checking `x in some_list` is **O(n)** because Python must scan
elements one by one until it finds a match or reaches the end. For 10,000
IDs checked repeatedly, the set is dramatically faster at scale.

##### Q41 - classify blood pressure

In [ ]:
def classify_bp(systolic):
    if systolic < 90:
        return "Low"
    elif systolic <= 120:
        return "Normal"
    elif systolic <= 139:
        return "Elevated"
    else:
        return "High"

for v in [85, 110, 130, 150]:
    print(v, "->", classify_bp(v))

85 -> Low
110 -> Normal
130 -> Elevated
150 -> High


#### Q42 - loop with break on first fever reading

In [ ]:
temps = [36.5, 36.8, 37.1, 39.8, 37.0]
for i, t in enumerate(temps):
    if t > 39.0:
        print(f"First fever at index {i}: {t}°C")
        break

First fever at index 3: 39.8°C


#### Q43 - sort by age descending using lambda

In [ ]:
patients_tuples = [("Ahmad", 34), ("Sara", 28), ("Bilal", 51)]
print(sorted(patients_tuples, key=lambda p: p[1], reverse=True))

[('Bilal', 51), ('Ahmad', 34), ('Sara', 28)]


#### Q44 - BMI function with default arg + type hints + docstring

In [ ]:
def bmi(weight: float, height: float, unit: str = "metric") -> float:
    """Return BMI. Metric: weight(kg) / height(m)**2."""
    return round(weight / height**2, 2)

print(bmi(70, 1.75))

22.86


#### Q45 - reproducible random list via seed + comprehension

In [ ]:
import random
random.seed(42)
rand_list = [random.randint(1, 100) for _ in range(5)]
print(rand_list)

[82, 15, 4, 95, 36]


**End of solution notebook.**